In [4]:
%pip install deltalake pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 2.3 MB/s  0:00:23m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


# NB02 · Semana 4 — Extracción de la API hacia la capa bronze

**Ingeniería de Datos — LCD · UCA Rosario · 2026**

Fuente del proyecto: **VATSIM** (`https://data.vatsim.net/v3/vatsim-data.json`), red de tráfico aéreo simulado. Gratis, sin API key, sin cuota.

## Objetivo

Construir el primer tramo del pipeline: **extraer** el snapshot de pilotos conectados a VATSIM y depositarlo en la capa **bronze** del lake como tabla Delta, aplicando dos patrones clave:

- **Carga incremental**: traer solo lo nuevo desde la última corrida (*watermark* sobre `snapshot_ts`).
- **Idempotencia**: reejecutar el pipeline no duplica datos ni rompe nada.

> **Nota sobre la fuente.** VATSIM no ofrece histórico ni filtro por fecha: cada llamada es una **foto del momento** (se actualiza cada ~15 s). El histórico se **construye acumulando snapshots** en bronze, una corrida = una foto, cada una con su `snapshot_ts`. Es un caso legítimo de ingesta incremental que genera la serie temporal.

> Este notebook es la base de la **Entrega 1 (E1)** del proyecto.

## 1. Configuración del acceso al lake

El mismo notebook corre en el Docker local y en el servidor de la cátedra: `entorno.py` (el de NB01) resuelve el bucket y las credenciales según el entorno.

In [5]:
from pathlib import Path
import os
import sys

# Buscar entorno.py desde la carpeta actual hacia arriba.
carpeta = Path.cwd().resolve()

for candidata in [carpeta, *carpeta.parents]:
    if (candidata / "entorno.py").is_file():
        ruta_entorno = candidata
        break
else:
    raise FileNotFoundError(
        "No se encontró entorno.py en la carpeta actual ni en sus superiores."
    )

sys.path.insert(0, str(ruta_entorno))

from entorno import BASE, BUCKET, STORAGE_OPTIONS, info

info()

# Tabla Delta de VATSIM dentro del bucket configurado para el grupo.
# No usamos BRONZE de entorno.py porque su valor por defecto es bronze/series.
BRONZE = f"s3://{BUCKET}/bronze/vatsim"
BRONZE_TEST = f"s3://{BUCKET}/bronze/_test_vatsim"

# Carpeta donde guardaremos el script de ingesta.
SCRIPTS_DIR = Path(BASE) / "scripts"

print("\nBronze de VATSIM:", BRONZE)
print("Bronze de pruebas:", BRONZE_TEST)
print("Carpeta de scripts:", SCRIPTS_DIR)

Entorno detectado: servidor de la cátedra  (grupo: grupo03)
  Lake / bucket   : grupo03   →   bronze en s3://grupo03/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo03', host 'postgres'
  Material en     : /home/jovyan/work

Bronze de VATSIM: s3://grupo03/bronze/vatsim
Bronze de pruebas: s3://grupo03/bronze/_test_vatsim
Carpeta de scripts: /home/jovyan/work/scripts


## 2. Leer el *watermark*

El watermark es el máximo `snapshot_ts` ya cargado en bronze. Si la tabla no existe todavía, devolvemos `None` (primera corrida).

Solo capturamos `TableNotFoundError`: cualquier otro error (MinIO caído, 403 por permisos) **debe fallar a la vista**, no hacerse pasar por una primera corrida.

In [6]:
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError


def leer_watermark(destino: str = BRONZE) -> str | None:
    """Devuelve el máximo snapshot_ts cargado en bronze."""
    try:
        dt = DeltaTable(destino, storage_options=STORAGE_OPTIONS)
    except TableNotFoundError:
        return None

    ts = dt.to_pandas(columns=["snapshot_ts"])["snapshot_ts"].dropna()
    return None if ts.empty else str(ts.max())


print("Watermark actual:", leer_watermark())

Watermark actual: 2026-10-16T22:53:54.0236694Z


## 3. Extraer de la API de VATSIM

Un solo endpoint, sin autenticación ni paginación: una llamada devuelve todos los pilotos conectados. El campo `general.update_timestamp` es el sello de tiempo del snapshot y hace de watermark.

In [7]:
import requests
import pandas as pd

BASE_URL = "https://data.vatsim.net/v3/vatsim-data.json"


def extraer_de_api(desde: str | None = None) -> pd.DataFrame:
    """Extrae el snapshot actual de pilotos conectados a VATSIM.

    VATSIM no tiene histórico ni filtro por fecha: cada llamada es una foto
    del momento. El histórico se construye acumulando snapshots en bronze,
    uno por corrida, cada uno identificado por 'snapshot_ts'.

    Args:
        desde: Watermark de la última corrida (update_timestamp ya ingerido).
            Si el snapshot actual no es más nuevo, se devuelve vacío para no
            reingerir la misma foto (idempotencia por watermark).

    Returns:
        DataFrame de pilotos, todo como string (bronze crudo; se tipa en silver),
        con la columna 'snapshot_ts'. El objeto anidado flight_plan se aplana
        a columnas flight_plan_* (queda nulo si el piloto no cargó plan).

    Raises:
        RuntimeError: Si VATSIM no responde correctamente.
    """
    try:
        r = requests.get(BASE_URL, timeout=30)
        r.raise_for_status()
    except requests.RequestException as exc:
        status = getattr(exc.response, "status_code", None)
        raise RuntimeError(f"Falló VATSIM ({type(exc).__name__}, HTTP {status})") from None

    data = r.json()
    snapshot_ts = data["general"]["update_timestamp"]
    if desde and snapshot_ts <= desde:
        print(f"Sin snapshot nuevo (watermark={desde}).")
        return pd.DataFrame()

    df = pd.json_normalize(data["pilots"], sep="_")
    df["snapshot_ts"] = snapshot_ts
    print(f"Filas extraídas: {len(df)} (snapshot {snapshot_ts})")
    return df.astype("string")


nuevos = extraer_de_api(desde=leer_watermark())
print(nuevos.shape); nuevos.head()

Filas extraídas: 909 (snapshot 2026-10-23T00:15:39.0236242Z)
(909, 34)


,cid,name,callsign,server,pilot_rating,military_rating,latitude,longitude,altitude,groundspeed,...,flight_plan_altitude,flight_plan_deptime,flight_plan_enroute_time,flight_plan_fuel_time,flight_plan_remarks,flight_plan_route,flight_plan_revision_id,flight_plan_assigned_transponder,flight_plan,snapshot_ts
0,1781530,Liam Boone KMCO,QFA1,USA-SE,0,0,-33.22856,153.06758,41195,469,...,24000,0325,2107,2255,PBN/A1B1C1D1L1O1S2 DOF/261022 REG/VHOJA EET/EG...,UMLAT T418 WELIN T420 TNT UN57 POL UN601 RIBEL...,1.0,0000,<NA>,2026-10-23T00:15:39.0236242Z
1,1998200,Matt J,BAW2620,UK,3,0,31.34206,-99.35286,33349,520,...,39000,0740,1652,1857,PBN/A1B1C1D1L1O1S2T1 NAV/RNVD1E2A1 SUR/TCAS DO...,HUH276027 TCC TXO LBB DILLO LAIKS4 KAUS,9.0,0675,<NA>,2026-10-23T00:15:39.0236242Z
2,921828,921828 EGLL,CAL012,UK,0,0,40.64097,-73.75875,28,11,...,19000,0940,1355,1604,PBN/A1B1C1D1L1O1S2 DOF/261022 REG/B18052 EET/R...,SAX070020 PUCKY PUCKY1 KJFK,7.0,7371,<NA>,2026-10-23T00:15:39.0236242Z
3,1419103,Gerasimos Stefanatos LGKF,FDX1988,GERMANY3,0,0,7.1846,157.86721,29957,500,...,30000,1815,1157,1351,PBN/A1B1C1D1L1O1S2 DOF/261022 REG/N605FE EET/N...,MOSMA4E MOSMA DCT TAF G224 KINAG/N0502F320 G22...,3.0,7775,<NA>,2026-10-23T00:15:39.0236242Z
4,1370888,M David YPDN,BAW11M,USA-WEST,0,0,1.36642,103.989,35,0,...,31000,1100,1253,1433,PBN/A1B1C1D1L1O1S2T1 NAV/RNP1 DAT/1FANSE2PDC S...,BPK Q295 BRAIN M197 REDFA DCT TACHA DCT SOGPO ...,5.0,3576,<NA>,2026-10-23T00:15:39.0236242Z


### Alternativa sin red: snapshot sintético

Para probar la ingesta y la idempotencia **sin depender del reloj** (VATSIM refresca cada ~15 s, así que dos corridas reales seguidas caen en fotos distintas). El `snapshot_ts` acá es **fijo**, así la segunda corrida siempre da 0. Se escribe en `BRONZE_TEST`.

In [8]:
def extraer_mock(desde: str | None = None) -> pd.DataFrame:
    """Snapshot VATSIM sintético y FIJO, para verificar idempotencia."""
    df = pd.DataFrame({
        "cid": ["1544123", "1472003", "1986825", "1962126", "1428384"],
        "callsign": ["DLH112", "UAE248", "SIA34", "KLM836", "QFA93"],
        "latitude": ["25.26", "25.24", "36.82", "52.31", "27.05"],
        "longitude": ["51.61", "55.35", "-121.85", "4.76", "-128.88"],
        "altitude": ["22", "27", "41052", "4", "39053"],
        "snapshot_ts": ["2026-09-25T22:00:00.0000000Z"] * 5,  # fijo a propósito
    }).astype("string")
    if desde and "2026-09-25T22:00:00.0000000Z" <= desde:
        return pd.DataFrame()
    return df


print("Snapshot sintético disponible para pruebas sin depender del reloj.")

Snapshot sintético disponible para pruebas sin depender del reloj.


## 4. Escribir en bronze (Delta, append idempotente)

Primera corrida: crea la tabla. Corridas siguientes: agrega solo snapshots nuevos.

**Clave natural:** `cid + snapshot_ts` — un piloto (identificado por su `cid`, único en VATSIM) en una foto dada. Dos barreras de idempotencia: el watermark evita traer la misma foto, y el anti-join por la clave evita duplicar aunque el snapshot se repita.

In [9]:
import uuid
from datetime import datetime, timezone

CLAVE = ["cid", "snapshot_ts"]


def ingerir(extractor, destino: str = BRONZE) -> int:
    """Ingiere en bronze solo las filas nuevas (incremental e idempotente).

    Args:
        extractor: Función que recibe 'desde' y devuelve un DataFrame de pilotos.
        destino: URI de la tabla Delta de bronze.

    Returns:
        Cantidad de filas escritas (0 si no había novedades).
    """
    wm = leer_watermark(destino)
    nuevos = extractor(desde=wm)
    if nuevos is None or nuevos.empty:
        print(f"Sin novedades (watermark={wm}).")
        return 0

    nuevos = nuevos.drop_duplicates(CLAVE)

    # Segunda barrera de idempotencia: descartar claves que ya existen en bronze.
    try:
        existentes = DeltaTable(destino, storage_options=STORAGE_OPTIONS).to_pandas(columns=CLAVE)
        claves_viejas = set(map(tuple, existentes.values))
        nuevos = nuevos[~nuevos[CLAVE].apply(tuple, axis=1).isin(claves_viejas)]
    except TableNotFoundError:
        pass  # primera corrida

    if nuevos.empty:
        print(f"Todo lo extraído ya estaba en bronze (watermark={wm}).")
        return 0

    nuevos = nuevos.assign(
        _ingested_at=datetime.now(timezone.utc).isoformat(),
        _run_id=uuid.uuid4().hex,
    )
    write_deltalake(destino, nuevos, mode="append", storage_options=STORAGE_OPTIONS)
    print(f"Ingeridas {len(nuevos)} filas (watermark previo={wm}).")
    return len(nuevos)


n = ingerir(extraer_de_api)
print("Filas ingeridas:", n)

Filas extraídas: 907 (snapshot 2026-10-23T00:16:39.0257877Z)
Ingeridas 907 filas (watermark previo=2026-10-16T22:53:54.0236694Z).
Filas ingeridas: 907


## 5. Verificar idempotencia

Con la fuente real no se puede demostrar de forma estable, porque VATSIM cambia el snapshot cada ~15 s: dos corridas seguidas caen en fotos distintas (y eso, lejos de ser un error, es cómo se acumula el histórico). Para demostrar la idempotencia usamos el snapshot **sintético y fijo** sobre `BRONZE_TEST`: la segunda corrida tiene que dar 0.

In [10]:
n1 = ingerir(extraer_mock, destino=BRONZE_TEST)
n2 = ingerir(extraer_mock, destino=BRONZE_TEST)
print(f"Corrida 1: {n1} filas | Corrida 2: {n2} filas")

df_test = DeltaTable(BRONZE_TEST, storage_options=STORAGE_OPTIONS).to_pandas()
dup = int(df_test.duplicated(CLAVE).sum())
print("Filas en bronze de prueba:", len(df_test), "| duplicadas:", dup)

assert n2 == 0, "La segunda corrida no debería agregar filas"
assert dup == 0, "Hay claves duplicadas en bronze"
print("Idempotencia verificada.")

Sin novedades (watermark=2026-09-25T22:00:00.0000000Z).
Sin novedades (watermark=2026-09-25T22:00:00.0000000Z).
Corrida 1: 0 filas | Corrida 2: 0 filas
Filas en bronze de prueba: 5 | duplicadas: 0
Idempotencia verificada.


## 6. Manejo de errores (documentación para la rúbrica)

La rúbrica de la E1 pide documentar el manejo de errores. Casos previstos:

| Situación | Cómo se maneja |
|---|---|
| VATSIM no responde / error HTTP | `extraer_de_api` levanta `RuntimeError` con el código, **antes** de tocar bronze; el watermark no avanza. |
| Snapshot sin cambios (misma foto) | El watermark detecta `snapshot_ts <= desde` y devuelve vacío: 0 filas, sin duplicar. |
| Snapshot repetido que igual llega a `ingerir` | El anti-join por `cid + snapshot_ts` descarta lo ya presente (segunda barrera). |
| Tabla inexistente (primera corrida) | Se captura solo `TableNotFoundError` → `None`; cualquier otro error (403, MinIO caído) **sí** se propaga. |
| Falla al escribir en bronze | El watermark se lee al inicio de la próxima corrida desde la tabla real, así que un fallo de escritura no lo deja desincronizado. |

**Trazabilidad:** cada fila lleva `_ingested_at` y `_run_id`, para linaje en bronze (append-only).

## 7. Guardar el script de ingesta para Airflow

In [11]:
from pathlib import Path
import inspect
import ast

carpeta_scripts = Path(BASE) / "scripts"
carpeta_scripts.mkdir(parents=True, exist_ok=True)

encabezado = '''from __future__ import annotations

import uuid
from datetime import datetime, timezone

import requests
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError

from entorno import BUCKET, STORAGE_OPTIONS

BRONZE = f"s3://{BUCKET}/bronze/vatsim"
BASE_URL = "https://data.vatsim.net/v3/vatsim-data.json"
CLAVE = ["cid", "snapshot_ts"]

'''

# Recuperar las funciones que acabamos de ejecutar y comprobar.
funciones = "\n\n".join(
    inspect.getsource(funcion)
    for funcion in [
        leer_watermark,
        extraer_de_api,
        ingerir,
    ]
)

ejecucion = '''
if __name__ == "__main__":
    filas = ingerir(extraer_de_api)
    print("Filas ingeridas:", filas)
'''

codigo = encabezado + funciones + "\n" + ejecucion

# Validar la sintaxis antes de escribir.
ast.parse(codigo)

ruta = carpeta_scripts / "ingesta.py"

# Conservar una copia si ya había un script diferente.
if ruta.exists() and ruta.read_text(encoding="utf-8") != codigo:
    respaldo = ruta.with_name(f"ingesta_{uuid.uuid4().hex[:8]}.bak")
    respaldo.write_bytes(ruta.read_bytes())
    print("Respaldo:", respaldo)

ruta.write_text(codigo, encoding="utf-8")
(carpeta_scripts / "__init__.py").touch(exist_ok=True)

print("Script guardado en:", ruta)

Script guardado en: /home/jovyan/work/scripts/ingesta.py


## Entrega 1 (E1)

Lo que se evalúa: ingesta idempotente hacia bronze con carga incremental y manejo de errores documentado.

**Pendiente:** confirmar con el profesor el cambio de fuente a VATSIM (tráfico simulado, sin histórico propio), porque afecta la ficha de dataset.


## Identificación de responsabilidades de la ingesta

Para la actividad integradora del grupo03, se identificaron las siguientes celdas del notebook de la clase 3 relacionadas con la extracción incremental de datos de VATSIM:

- **Lectura de configuración y credenciales:** celda 5. Importación de la configuración desde `entorno.py` y definición de las rutas de almacenamiento.
- **Llamada a la API:** celda 9. Consulta HTTP a la API pública de VATSIM mediante `requests.get()`.
- **Validación de la respuesta:** celda 9. Verificación del estado HTTP con `raise_for_status()` y manejo de errores de conexión.
- **Normalización o transformación de registros:** celda 9. Transformación del JSON de pilotos mediante `pd.json_normalize()` y conversión de tipos.
- **Watermark o criterio incremental:** celdas 7 y 13. Lectura del último snapshot almacenado y utilización de esa marca para identificar nuevos datos.
- **Escritura en la capa Bronze:** celda 13. Almacenamiento de los registros en Delta Lake mediante `write_deltalake()`.
- **Mensajes de control y manejo de errores:** celdas 5, 7, 9 y 13. Comprobaciones, mensajes informativos y tratamiento de excepciones.

La lógica reutilizable se encuentra implementada en `scripts/ingesta.py`, separando la extracción desde la API, el control incremental y la escritura en Bronze.

Las referencias corresponden a la numeración de celdas utilizada durante el análisis del notebook..
